# Exploration ACJEUNES - Coupe de France Jeunes

Analyse de la structure des pages pour découvrir tous les matchs disponibles.

In [ ]:
import requests
from bs4 import BeautifulSoup
import re
from collections import defaultdict

# Fetch the ACJEUNES home page
session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36'
})

url = 'https://www.ffvbbeach.org/ffvbapp/resu/vbspo_home.php?saison=2025/2026&codent=ACJEUNES'
resp = session.get(url, timeout=30)
soup = BeautifulSoup(resp.content, 'html.parser')

# Extract ALL links with their full hrefs
all_links = []
for link in soup.find_all('a', href=True):
    href = link['href']
    text = link.get_text(strip=True)
    if 'vbspo_calendrier' in href or 'division=' in href or 'poule=' in href or 'index_' in href:
        all_links.append({'href': href, 'text': text})

print(f'Found {len(all_links)} relevant links')
for l in all_links[:40]:
    print(f'  {l["text"][:60]:60s} -> {l["href"]}')

In [ ]:
# Extract ALL division codes and poule codes
divisions = set()
poules = set()
comp_structure = defaultdict(list)

for link in soup.find_all('a', href=True):
    href = link['href']
    text = link.get_text(strip=True)
    
    div_match = re.search(r'division=([^&]+)', href)
    poule_match = re.search(r'poule=([^&]+)', href)
    
    if div_match:
        code = div_match.group(1)
        divisions.add(code)
        comp_structure[code].append(text)
    if poule_match:
        code = poule_match.group(1)
        poules.add(code)
        comp_structure[code].append(text)

print(f'Divisions found: {len(divisions)}')
for d in sorted(divisions):
    print(f'  {d}: {comp_structure[d][:3]}')

print(f'\nPoules found: {len(poules)}')
for p in sorted(poules):
    print(f'  {p}: {comp_structure[p][:3]}')

In [ ]:
# Now let's look at ONE division calendar page to understand match structure
# Pick the first division code
sample_division = sorted(divisions)[0] if divisions else sorted(poules)[0] if poules else None
print(f'Sampling division: {sample_division}')

if sample_division:
    cal_url = f'https://www.ffvbbeach.org/ffvbapp/resu/vbspo_calendrier.php?saison=2025/2026&codent=ACJEUNES&division={sample_division}&calend=COMPLET'
    print(f'URL: {cal_url}')
    resp2 = session.get(cal_url, timeout=30)
    soup2 = BeautifulSoup(resp2.content, 'html.parser')
    
    # Count forms with match links
    forms = soup2.find_all('form')
    match_count = 0
    match_codes = []
    for form in forms:
        action = form.get('action', '')
        if 'ffvolley_fdme.php' in action:
            m = re.search(r'codmatch=([^&]+)', action)
            if m:
                match_count += 1
                match_codes.append(m.group(1))
        elif action.endswith('.pdf'):
            match_count += 1
            match_codes.append(action.split('/')[-1])
    
    print(f'Matches found: {match_count}')
    for mc in match_codes[:20]:
        print(f'  {mc}')

In [ ]:
# Test inaccessible matches
# LIIDF/RMT 2023/2024, PTRA42/CMP 2023/2024, LIFL/CMP 2024/2025
test_cases = [
    ('LIIDF', 'RMT', '2023/2024', False),
    ('PTRA42', 'CMP', '2023/2024', False),
    ('LIFL', 'CMP', '2024/2025', False),
]

for entity_code, poule_code, saison, is_division in test_cases:
    # Try both poule= and division= URL formats
    for param_name in ['poule', 'division']:
        url = f'https://www.ffvbbeach.org/ffvbapp/resu/vbspo_calendrier.php?saison={saison}&codent={entity_code}&{param_name}={poule_code}&calend=COMPLET'
        print(f'\nTesting {entity_code}/{poule_code} ({saison}) with {param_name}=')
        print(f'  URL: {url}')
        try:
            resp = session.get(url, timeout=30)
            print(f'  Status: {resp.status_code}')
            soup_test = BeautifulSoup(resp.content, 'html.parser')
            
            # Check for match forms
            forms = soup_test.find_all('form')
            match_forms = [f for f in forms if 'ffvolley_fdme.php' in f.get('action', '')]
            print(f'  Match forms: {len(match_forms)}')
            
            # Check title/header text
            title = soup_test.find('title')
            if title:
                print(f'  Page title: {title.text.strip()[:100]}')
            
            # Look for any relevant content
            body_text = soup_test.get_text()
            if 'pas de calendrier' in body_text.lower() or 'aucun' in body_text.lower():
                print('  -> Page indicates no calendar available')
            
            # Check if page has any tables with match data
            tables = soup_test.find_all('table')
            print(f'  Tables found: {len(tables)}')
            
        except Exception as e:
            print(f'  Error: {e}')

In [ ]:
# Check what poules exist for these entities
for entity_code, saison in [('LIIDF', '2023/2024'), ('PTRA42', '2023/2024'), ('LIFL', '2024/2025')]:
    url = f'https://www.ffvbbeach.org/ffvbapp/resu/vbspo_home.php?saison={saison}&codent={entity_code}'
    print(f'\n=== {entity_code} - {saison} ===')
    print(f'URL: {url}')
    try:
        resp = session.get(url, timeout=30)
        soup_home = BeautifulSoup(resp.content, 'html.parser')
        
        # Find links containing CMP or RMT
        for link in soup_home.find_all('a', href=True):
            href = link['href']
            text = link.get_text(strip=True)
            code = ''
            m = re.search(r'poule=([^&]+)', href)
            if m:
                code = m.group(1)
            m2 = re.search(r'division=([^&]+)', href)
            if m2:
                code = m2.group(1)
            
            if code and ('CMP' in code.upper() or 'RMT' in code.upper() or text and any(k in text.upper() for k in ['COUPE', 'CHALLENGE', 'RMT'])):
                print(f'  Found: {text} -> {code} ({href})')
    except Exception as e:
        print(f'  Error: {e}')